<td width="45%" valign="middle">
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="display:inline-block; vertical-align:middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="display:inline-block; vertical-align:middle;"
    >
  </div>
</td>

<td width="55%" align="right" valign="middle">
  <p style="margin: 0;"><b>Intelligence Data Science and Artificial Intelligence (IDEAI)</b></p>
  <p style="margin: 0;"><b>Grau en Estadística (UB - UPC)</b></p>
  <p style="margin: 0;">Anàlisis Multivariant de Dades (AMD)</p>
</td>


# **Support Vector Machine (SVM)**

SVM lineal y SVM con kernel RBF.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay
)

csv_path = "dataset_bancario_sintetico.csv"
df = pd.read_csv(csv_path)

print("Dimensiones:", df.shape)
display(df.head())
display(df.info())

print("\nDistribución de default:")
display(df["default"].value_counts(dropna=False))
display(df["default"].value_counts(normalize=True).rename("proporcion"))

In [ ]:
columnas_a_excluir = ["default", "cliente_id"]
columnas_presentes_a_excluir = [c for c in columnas_a_excluir if c in df.columns]

X = df.drop(columns=columnas_presentes_a_excluir)
y = df["default"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

variables_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
variables_categoricas = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nVariables numéricas:")
print(variables_numericas)
print("\nVariables categóricas:")
print(variables_categoricas)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("\nTamaño train:", X_train.shape)
print("Tamaño test :", X_test.shape)

In [ ]:
transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", transformador_numerico, variables_numericas),
    ("cat", transformador_categorico, variables_categoricas)
])

In [ ]:
from sklearn.svm import SVC

modelo_svm_lineal = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(kernel="linear", C=1.0, probability=True, random_state=42))
])
modelo_svm_lineal.fit(X_train, y_train)
pred_lineal = modelo_svm_lineal.predict(X_test)
proba_lineal = modelo_svm_lineal.predict_proba(X_test)[:, 1]

modelo_svm_rbf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42))
])
modelo_svm_rbf.fit(X_train, y_train)
pred_rbf = modelo_svm_rbf.predict(X_test)
proba_rbf = modelo_svm_rbf.predict_proba(X_test)[:, 1]

tabla_svm = pd.DataFrame([
    {"modelo": "SVM lineal", "auc": roc_auc_score(y_test, proba_lineal), "f1": f1_score(y_test, pred_lineal)},
    {"modelo": "SVM RBF", "auc": roc_auc_score(y_test, proba_rbf), "f1": f1_score(y_test, pred_rbf)}
])
display(tabla_svm)